In [0]:
# 1. CẤU HÌNH VÀ TẠO MỚI CATALOG & SCHEMA THEO CHUẨN UNITY CATALOG
CATALOG_NAME = "omni_retail_catalog"  # Bạn có thể đổi tên Catalog tùy ý
SCHEMA_NAME = "bronze"

# Tạo và kích hoạt Catalog mới
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
spark.sql(f"USE CATALOG {CATALOG_NAME}")

# Tạo và kích hoạt Schema Bronze bên trong Catalog
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA_NAME}")
spark.sql(f"USE SCHEMA {SCHEMA_NAME}")

print(f"Đang làm việc tại: {CATALOG_NAME}.{SCHEMA_NAME}")

# 2. INGEST BẢNG SALES ORDERS (JSON) - XỬ LÝ LỖI METADATA TRÊN UNITY CATALOG
from pyspark.sql import functions as F

source_sales = "dbfs:/databricks-datasets/retail-org/sales_orders/"
target_sales_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.sales_orders"

df_sales_raw = spark.read.json(source_sales)

(
    df_sales_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))  # Cú pháp chuẩn Unity Catalog
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_sales_table)
)

print(f"Hoàn thành Ingest Sales: {spark.table(target_sales_table).count():,} dòng.")

# 3. INGEST BẢNG CUSTOMERS MASTER DATA (CSV)
source_cust = "dbfs:/databricks-datasets/retail-org/customers/"
target_cust_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.customers"

df_cust_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_cust)
)

(
    df_cust_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_cust_table)
)

print(f"Hoàn thành Ingest Customers: {spark.table(target_cust_table).count():,} khách hàng.")